# Allen Cell Types — Dataset Summary

**Dataset:** Allen Institute Cell Types Database (CRCNS)
**Species:** Mus musculus — 1,920 cells, whole-cell patch clamp, primary visual cortex and surrounding areas

**Purpose:** Ground-truth cell type reference for validating spikeparam waveform features.
Known labels (dendrite type, transgenic line, cortical layer, brain area) allow us to confirm
that our waveform features separate cell types consistently with established markers.

**Key label mapping:**
- `spiny`          → pyramidal cell (PC)
- `aspiny`         → interneuron (IN)
- `sparsely spiny` → ambiguous / unclassified


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sys.path.append('../datasets/allen-cell-types/allen_ct_helper_modules/')
from data_loader import load_cell_metadata, load_precomputed_features, load_morphology_features
from config import ALLEN_CT_PICKLE_ROOT

# Cell-type colours matching spe-1 / pvc-6 conventions
COL_SPINY    = '#CC44CC'   # magenta  — spiny (PC)
COL_ASPINY   = '#00CCCC'   # cyan     — aspiny (IN)
COL_SPARSE   = '#AAAAAA'   # grey     — sparsely spiny

DEND_COLS = {
    'spiny':          COL_SPINY,
    'aspiny':         COL_ASPINY,
    'sparsely spiny': COL_SPARSE,
}

FS_TITLE = 11
FS_LABEL = 10
FS_TICK  = 9
FS_ANNOT = 8

In [ ]:
# ── 1. Metadata ───────────────────────────────────────────────────────────────
meta = load_cell_metadata(species='Mus musculus')
print(f'n = {len(meta)} cells')
print()
print('Dendrite type:')
print(meta['dendrite_type'].value_counts().to_string())
print()
print('Brain area (top 10):')
print(meta['structure_area_abbrev'].value_counts().head(10).to_string())
print()
print('Cortical layer:')
print(meta['structure_layer_name'].value_counts().to_string())
print()
print('Transgenic lines (top 10):')
print(meta['transgenic_line'].value_counts().head(10).to_string())
print()

# ── 2. Precomputed ephys features ─────────────────────────────────────────────
feat_df = load_precomputed_features()
merged_feat = meta.merge(feat_df, left_on='id', right_index=True, how='left')
n_ephys = merged_feat[feat_df.columns[0]].notna().sum()
print(f'Ephys features: {len(feat_df.columns)} columns, {n_ephys}/{len(meta)} cells matched')

# ── 3. Morphology features ────────────────────────────────────────────────────
morph_df = load_morphology_features()
merged_morph = meta.merge(morph_df, left_on='id', right_index=True, how='left')
_morph_check = 'total_length' if 'total_length' in morph_df.columns else morph_df.columns[0]
n_morph = merged_morph[_morph_check].notna().sum()
print(f'Morphology features: {len(morph_df.columns)} columns, {n_morph}/{len(meta)} cells with reconstructions')

In [ ]:
# ── Figure: dataset summary ────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.50, wspace=0.42)

DEND_ORDER = ['spiny', 'aspiny', 'sparsely spiny']

# ── A. Dendrite type breakdown ─────────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
dt_counts = meta['dendrite_type'].value_counts().reindex(DEND_ORDER).dropna()
bars = ax_a.bar(dt_counts.index, dt_counts.values,
                color=[DEND_COLS[d] for d in dt_counts.index],
                edgecolor='white', width=0.55)
for bar, val in zip(bars, dt_counts.values):
    ax_a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
              str(int(val)), ha='center', va='bottom', fontsize=FS_ANNOT, fontweight='bold')
ax_a.set_ylabel('N cells', fontsize=FS_LABEL)
ax_a.set_title('A.  Dendrite type', fontsize=FS_TITLE, fontweight='bold', loc='left')
ax_a.set_xticklabels(['spiny\n(PC)', 'aspiny\n(IN)', 'sparsely\nspiny'], fontsize=FS_TICK)
ax_a.set_ylim(0, dt_counts.max() * 1.25)
sns.despine(ax=ax_a)

# ── B. Brain area (top 10) ─────────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
area_counts = meta['structure_area_abbrev'].value_counts().head(10)
ax_b.barh(area_counts.index[::-1], area_counts.values[::-1],
          color='#5b83bc', edgecolor='white', alpha=0.85)
for i, (area, val) in enumerate(zip(area_counts.index[::-1], area_counts.values[::-1])):
    ax_b.text(val + 5, i, str(val), va='center', fontsize=FS_ANNOT)
ax_b.set_xlabel('N cells', fontsize=FS_LABEL)
ax_b.set_title('B.  Brain area (top 10)', fontsize=FS_TITLE, fontweight='bold', loc='left')
ax_b.tick_params(axis='y', labelsize=FS_TICK)
sns.despine(ax=ax_b)

# ── C. Cortical layer by dendrite type ─────────────────────────────────────────
ax_c = fig.add_subplot(gs[0, 2])
LAYER_ORDER = ['1', '2/3', '4', '5', '6a', '6b']
layer_dt = (meta.groupby(['structure_layer_name', 'dendrite_type'])
              .size().unstack(fill_value=0)
              .reindex(LAYER_ORDER, fill_value=0))
bot = np.zeros(len(layer_dt))
x   = np.arange(len(layer_dt))
for dend in ['spiny', 'aspiny', 'sparsely spiny']:
    if dend not in layer_dt.columns:
        continue
    vals = layer_dt[dend].values.astype(float)
    ax_c.bar(x, vals, bottom=bot, color=DEND_COLS[dend],
             edgecolor='white', alpha=0.9, label=dend, width=0.6)
    bot += vals
ax_c.set_xticks(x)
ax_c.set_xticklabels(LAYER_ORDER, fontsize=FS_TICK)
ax_c.set_ylabel('N cells', fontsize=FS_LABEL)
ax_c.set_xlabel('Cortical layer', fontsize=FS_LABEL)
ax_c.set_title('C.  Layer distribution', fontsize=FS_TITLE, fontweight='bold', loc='left')
ax_c.legend(fontsize=FS_ANNOT, frameon=False, loc='upper left')
sns.despine(ax=ax_c)

# ── D. Transgenic lines (top 10, coloured by known cell type) ──────────────────
ax_d = fig.add_subplot(gs[0, 3])
# Known IN lines: Pvalb, Sst, Htr3a, Vip, Ndnf; PC lines: Rorb, Nr5a1, Scnn1a, Rbp4, Cux2
IN_LINES = {'Pvalb', 'Sst', 'Htr3a', 'Vip', 'Ndnf'}
top_lines = meta['transgenic_line'].value_counts().head(10)
line_cols = [COL_ASPINY if any(il in ln for il in IN_LINES) else COL_SPINY
             for ln in top_lines.index]
ax_d.barh(top_lines.index[::-1], top_lines.values[::-1],
          color=line_cols[::-1], edgecolor='white', alpha=0.85)
for i, val in enumerate(top_lines.values[::-1]):
    ax_d.text(val + 2, i, str(val), va='center', fontsize=FS_ANNOT)
ax_d.set_xlabel('N cells', fontsize=FS_LABEL)
ax_d.set_title('D.  Transgenic line (top 10)', fontsize=FS_TITLE, fontweight='bold', loc='left')
ax_d.tick_params(axis='y', labelsize=FS_TICK)
sns.despine(ax=ax_d)

# ── E. Normalized depth by dendrite type ──────────────────────────────────────
ax_e = fig.add_subplot(gs[1, 0:2])
rng = np.random.default_rng(42)
for i, dend in enumerate(['spiny', 'aspiny', 'sparsely spiny']):
    sub = meta[meta['dendrite_type'] == dend]['normalized_depth'].dropna()
    jitter = rng.uniform(-0.1, 0.1, len(sub))
    ax_e.scatter(i + jitter, sub.values,
                 color=DEND_COLS[dend], s=12, alpha=0.4,
                 edgecolors='none', zorder=3)
    ax_e.plot([i - 0.25, i + 0.25], [sub.median()] * 2,
              color=DEND_COLS[dend], lw=2.5, zorder=4)
ax_e.set_xticks([0, 1, 2])
ax_e.set_xticklabels(['spiny (PC)', 'aspiny (IN)', 'sparsely spiny'], fontsize=FS_TICK)
ax_e.set_ylabel('Normalised cortical depth', fontsize=FS_LABEL)
ax_e.set_title('E.  Cortical depth by dendrite type', fontsize=FS_TITLE, fontweight='bold', loc='left')
ax_e.invert_yaxis()
ax_e.set_ylim(1.05, -0.05)
sns.despine(ax=ax_e)

# ── F. Dendrite type × layer heatmap ─────────────────────────────────────────
ax_f = fig.add_subplot(gs[1, 2:4])
heat = (meta.groupby(['structure_layer_name', 'dendrite_type'])
           .size().unstack(fill_value=0)
           .reindex(LAYER_ORDER))
pct  = heat.div(heat.sum(axis=1), axis=0) * 100
sns.heatmap(pct[['spiny', 'aspiny', 'sparsely spiny']],
            annot=True, fmt='.0f', cmap='RdPu',
            linewidths=0.5, ax=ax_f,
            cbar_kws={'label': '% of cells in layer'},
            annot_kws={'size': FS_ANNOT})
ax_f.set_xlabel('Dendrite type', fontsize=FS_LABEL)
ax_f.set_ylabel('Cortical layer', fontsize=FS_LABEL)
ax_f.set_title('F.  Dendrite type × layer (% per layer)',
               fontsize=FS_TITLE, fontweight='bold', loc='left')
ax_f.tick_params(axis='both', labelsize=FS_TICK)

n_spiny  = (meta['dendrite_type'] == 'spiny').sum()
n_aspiny = (meta['dendrite_type'] == 'aspiny').sum()
n_sparse = (meta['dendrite_type'] == 'sparsely spiny').sum()
fig.suptitle(
    f'Allen Cell Types — mouse — n = {len(meta)} cells  '
    f'({n_spiny} spiny / {n_aspiny} aspiny / {n_sparse} sparsely spiny)  '
    f'|  whole-cell patch clamp  |  VISp + surrounding areas',
    fontsize=12, fontweight='bold', y=1.01
)

plt.savefig('allen_ct_dataset_summary.pdf', bbox_inches='tight', dpi=300)
plt.savefig('allen_ct_dataset_summary.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved.')

## 2. Precomputed Ephys Features

Allen precomputed features from Long Square step current injection sweeps.
These serve as ground truth for validating spikeparam waveform separations.

Key discriminators between aspiny (IN) and spiny (PC):
- **Upstroke/downstroke ratio**: aspiny ~1.9 vs spiny ~3.5 (narrow fast spikes in INs)
- **Avg ISI**: aspiny ~32 ms vs spiny ~80 ms (higher firing rates in INs)
- **Membrane τ**: aspiny ~10 ms vs spiny ~20 ms (faster membrane in INs)

In [ ]:
# ── Figure 2: Precomputed ephys features by dendrite type ─────────────────────
DEND_ORDER_2 = ['spiny', 'aspiny']
DEND_COLS_2  = [COL_SPINY, COL_ASPINY]

EPHYS_PANELS = [
    ('upstroke_downstroke_ratio_long_square', 'G', 'Upstroke/downstroke ratio'),
    ('avg_isi',                               'H', 'Avg ISI (ms)'),
    ('tau',                                   'I', 'Membrane τ (ms)'),
    ('threshold_v_long_square',               'J', 'Threshold voltage (mV)'),
    ('vrest',                                 'K', 'Resting Vm (mV)'),
    ('adaptation',                            'L', 'ISI adaptation'),
]
# Keep only panels whose column is present
EPHYS_PANELS = [(c, lbl, name) for c, lbl, name in EPHYS_PANELS if c in merged_feat.columns]

sub_feat = merged_feat[merged_feat['dendrite_type'].isin(DEND_ORDER_2)].copy()

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes_flat = list(axes.flat)

rng = np.random.default_rng(42)
for ax, (col, panel_lbl, ylabel) in zip(axes_flat, EPHYS_PANELS):
    n_sp = sub_feat[sub_feat['dendrite_type'] == 'spiny'][col].dropna().shape[0]
    n_as = sub_feat[sub_feat['dendrite_type'] == 'aspiny'][col].dropna().shape[0]
    for i, (dt, c) in enumerate(zip(DEND_ORDER_2, DEND_COLS_2)):
        vals = sub_feat[sub_feat['dendrite_type'] == dt][col].dropna().values
        if len(vals) < 5:
            continue
        parts = ax.violinplot(vals, positions=[i], widths=0.65,
                              showmedians=False, showextrema=False)
        for pc in parts['bodies']:
            pc.set_facecolor(c)
            pc.set_alpha(0.35)
            pc.set_linewidth(0.5)
        jit = rng.uniform(-0.12, 0.12, len(vals))
        ax.scatter(i + jit, vals, color=c, s=3, alpha=0.5,
                   linewidths=0, zorder=3)
        ax.plot([i - 0.18, i + 0.18], [np.median(vals)] * 2,
                'k-', lw=2.2, zorder=5)
    ax.set_xticks([0, 1])
    ax.set_xticklabels([f'spiny\n(PC)\nn={n_sp}', f'aspiny\n(IN)\nn={n_as}'],
                       fontsize=FS_TICK)
    ax.set_ylabel(ylabel, fontsize=FS_LABEL)
    ax.set_title(f'{panel_lbl}.  {ylabel}', fontsize=FS_TITLE,
                 fontweight='bold', loc='left')
    ax.set_xlim(-0.6, 1.6)
    sns.despine(ax=ax)

for ax in axes_flat[len(EPHYS_PANELS):]:
    ax.set_visible(False)

_n_sp_all = sub_feat[sub_feat['dendrite_type'] == 'spiny'].shape[0]
_n_as_all = sub_feat[sub_feat['dendrite_type'] == 'aspiny'].shape[0]
fig.suptitle(
    f'Allen Cell Types — precomputed ephys features (Long Square)  '
    f'|  spiny n={_n_sp_all}  aspiny n={_n_as_all}',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('allen_ct_ephys_features.pdf', bbox_inches='tight', dpi=300)
plt.savefig('allen_ct_ephys_features.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved.')

## 3. Morphology Features

Morphological features from 3D neuron reconstructions (~537 mouse cells with reconstructions).
Aspiny (IN) cells have shorter total neurite length and fewer bifurcations than spiny (PC) cells,
consistent with their compact local-circuit morphology relative to long-range pyramidal cells.

In [ ]:
# ── Figure 3: Morphology features by dendrite type ────────────────────────────
MORPH_PANELS = [
    ('total_length',          'M', 'Total neurite length (µm)'),
    ('soma_surface',          'N', 'Soma surface area (µm²)'),
    ('number_bifurcations',   'O', 'Number of bifurcations'),
    ('max_euclidean_distance','P', 'Max Euclidean distance (µm)'),
]
# Keep only columns present in morph_df
MORPH_PANELS = [(c, lbl, name) for c, lbl, name in MORPH_PANELS if c in morph_df.columns]

sub_morph = merged_morph[merged_morph['dendrite_type'].isin(DEND_ORDER_2)].copy()

n_cols = min(4, len(MORPH_PANELS))
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 5))
if n_cols == 1:
    axes = [axes]

rng2 = np.random.default_rng(99)
for ax, (col, panel_lbl, ylabel) in zip(axes, MORPH_PANELS):
    for i, (dt, c) in enumerate(zip(DEND_ORDER_2, DEND_COLS_2)):
        vals = sub_morph[sub_morph['dendrite_type'] == dt][col].dropna().values
        if len(vals) < 5:
            continue
        parts = ax.violinplot(vals, positions=[i], widths=0.65,
                              showmedians=False, showextrema=False)
        for pc in parts['bodies']:
            pc.set_facecolor(c)
            pc.set_alpha(0.35)
            pc.set_linewidth(0.5)
        jit = rng2.uniform(-0.12, 0.12, len(vals))
        ax.scatter(i + jit, vals, color=c, s=4, alpha=0.5,
                   linewidths=0, zorder=3)
        ax.plot([i - 0.18, i + 0.18], [np.median(vals)] * 2,
                'k-', lw=2.2, zorder=5)
    ax.set_xticks([0, 1])
    n_sp_m = sub_morph[sub_morph['dendrite_type'] == 'spiny'][col].dropna().shape[0]
    n_as_m = sub_morph[sub_morph['dendrite_type'] == 'aspiny'][col].dropna().shape[0]
    ax.set_xticklabels([f'spiny\n(PC)\nn={n_sp_m}', f'aspiny\n(IN)\nn={n_as_m}'],
                       fontsize=FS_TICK)
    ax.set_ylabel(ylabel, fontsize=FS_LABEL)
    ax.set_title(f'{panel_lbl}.  {ylabel}', fontsize=FS_TITLE,
                 fontweight='bold', loc='left')
    ax.set_xlim(-0.6, 1.6)
    sns.despine(ax=ax)

n_morph_sp = sub_morph[sub_morph['dendrite_type'] == 'spiny'].shape[0]
n_morph_as = sub_morph[sub_morph['dendrite_type'] == 'aspiny'].shape[0]
fig.suptitle(
    f'Allen Cell Types — morphology features (cells with reconstructions)  '
    f'|  spiny n={n_morph_sp}  aspiny n={n_morph_as}',
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('allen_ct_morph_features.pdf', bbox_inches='tight', dpi=300)
plt.savefig('allen_ct_morph_features.png', bbox_inches='tight', dpi=300)
plt.show()
print('Saved.')